Below is example code used to fine-tune the Detectron2 pre-trained segmentation base model. Please provide your own folder names and registered datasets.

### ⚠️ Google Colab Only Notebook

This notebook is designed to run in **Google Colab**. To start using this notebook, clone this GitHub repo in Colab:
```
!git clone https://github.com/Lynnicia/CryoEM_ultrastructures_top_model_decision_tree/tree/main.git
```
It requires:
- GPU runtime (CUDA) for speed tests
- Linux environment
- Colab-specific install steps (e.g., `!pip install`, `!git clone`, `from google.colab.patches import cv2_imshow`)

❌ Running locally (especially on Windows) may fail due to:
- Detectron2 installation issues
- CUDA/toolchain mismatches
- Missing compiled operators

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install roboflow --no-deps
!pip install pillow
!apt-get update
!apt-get install -y libavif-dev libaom-dev
!pip install --no-cache-dir pillow-avif-plugin pi-heif
!pip install filetype

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="[your_api_key]")
project = rf.workspace("[workspace_name]").project("[project_name]")
version = project.version([version_number])
dataset = version.download("coco-segmentation")

In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
#copy jsons into the main folder for Detectron2
!cp /content/[roboflow_project_folder_name]/train/_annotations.coco.json /content/[roboflow_project_folder_name]/t.json

!cp /content/[roboflow_project_folder_name]/valid/_annotations.coco.json /content/[roboflow_project_folder_name]/v.json


In [ ]:
#inspect json format for TRAIN and delete bacteria category to only keep OM and IM categories

import json

with open("/content/[roboflow_project_folder_name]/t.json") as f:
    coco = json.load(f)

for c in coco["categories"]:
    print(c["id"], c["name"])

cats = sorted(coco["categories"], key=lambda x: x["id"])


# Update annotations

BAD_ID = 0

coco["categories"] = [
    c for c in coco["categories"] if c["id"] != BAD_ID
]

coco["annotations"] = [
    ann for ann in coco["annotations"]
    if ann["category_id"] != BAD_ID
]

with open("/content/[roboflow_project_folder_name]/train.json", "w") as f:
    json.dump(coco, f, indent=2)

for c in coco["categories"]:
    print(c["id"], c["name"])


In [ ]:
#inspect json format for VALID and delete bacteria category

import json

with open("/content/[roboflow_project_folder_name]/v.json") as f:
    coco = json.load(f)

for c in coco["categories"]:
    print(c["id"], c["name"])

cats = sorted(coco["categories"], key=lambda x: x["id"])


# Update annotations

BAD_ID = 0

coco["categories"] = [
    c for c in coco["categories"] if c["id"] != BAD_ID
]

coco["annotations"] = [
    ann for ann in coco["annotations"]
    if ann["category_id"] != BAD_ID
]

with open("/content/[roboflow_project_folder_name]/valid.json", "w") as f:
    json.dump(coco, f, indent=2)

for c in coco["categories"]:
    print(c["id"], c["name"])


In [ ]:
from detectron2.engine import HookBase
from detectron2.evaluation import inference_on_dataset, COCOEvaluator
from detectron2.data import build_detection_test_loader

class APHook(HookBase):
    def __init__(self, cfg, dataset_name, period=200):
        self.cfg = cfg
        self.dataset_name = dataset_name
        self.period = period
        self.best_ap = -1.0
        self.last_ap = None

    def after_step(self):
        it = self.trainer.iter
        if it % self.period != 0:
            return

        model = self.trainer.model
        model.eval()

        evaluator = COCOEvaluator(
            self.dataset_name,
            distributed=False,
            output_dir=self.cfg.OUTPUT_DIR,
        )

        val_loader = build_detection_test_loader(
            self.cfg,
            self.dataset_name
        )

        results = inference_on_dataset(model, val_loader, evaluator)

        ap = results["segm"]["AP"]
        self.last_ap = ap

        print(f"[VAL] iter={it} segm_AP={ap:.4f}")

        if ap > self.best_ap:
            self.best_ap = ap
            self.trainer.checkpointer.save("best_model")
            print(f"New best model saved (AP={ap:.4f})")

        model.train()

In [ ]:
from detectron2.engine import HookBase

class EarlyStoppingHook(HookBase):
    def __init__(self, ap_hook, patience=5, period=200, min_delta=0.001):
        self.ap_hook = ap_hook
        self.patience = patience
        self.period = period
        self.min_delta = min_delta

        self.best = -1.0
        self.counter = 0

    def after_step(self):
        it = self.trainer.iter

        # Only check at evaluation steps
        if it % self.period != 0:
            return

        val_ap = self.ap_hook.last_ap
        if val_ap is None:
            return

        print(f"[EARLYSTOP] iter={it} segm_AP={val_ap:.6f}")

        # Improvement
        if val_ap > (self.best + self.min_delta):
            self.best = val_ap
            self.counter = 0
            print("Improved. Counter reset.")
        else:
            self.counter += 1
            print(f"No improvement. Patience: {self.counter}/{self.patience}")

        if self.counter >= self.patience:
            print("Early stopping triggered ✅")
            raise StopIteration

In [ ]:
# function to globally register dataset in Detectron 2

from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances

def reregister_coco(name, json_file, image_root):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)
        MetadataCatalog.remove(name)

    register_coco_instances(name, {}, json_file, image_root)

In [ ]:
# ALWAYS rename dataset if data changes!!

for d in ["train", "valid"]:
    reregister_coco(
        f"[your_sample_name]_{d}",
        f"/content/[roboflow_project_folder_name]/{d}.json",
        f"/content/[roboflow_project_folder_name]/{d}"
    )

In [ ]:
td = DatasetCatalog.get("[your_sample_name]_train")[0]
print(td)
vd = DatasetCatalog.get("[your_sample_name]_valid")[0]
print(vd)

In [ ]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [ ]:
##################### Train the model ##########################

# Change to a known accessible directory
%cd /content/bacteria-thickness_additional-6

from detectron2.data import DatasetCatalog
from detectron2.engine import DefaultTrainer
from detectron2.config import get_cfg
import pandas as pd


cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")) # removed normalization, standard configuration used: https://deepwiki.com/facebookresearch/detectron2/7-model-zoo
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
cfg.MODEL.PIXEL_MEAN = [0.0, 0.0, 0.0] # disable normalization
cfg.MODEL.PIXEL_STD  = [1.0, 1.0, 1.0] # disable normalization
cfg.INPUT.MIN_SIZE_TRAIN = (1024,)
cfg.INPUT.MAX_SIZE_TRAIN = 1024
cfg.INPUT.MIN_SIZE_TEST = 1024
cfg.INPUT.MAX_SIZE_TEST = 1024
cfg.DATASETS.TRAIN = ("[your_sample_name]_train",)
cfg.DATASETS.TEST = ("[your_sample_name]_valid",)
cfg.INPUT.MASK_FORMAT = "bitmask"
cfg.SOLVER.CHECKPOINT_PERIOD = cfg.SOLVER.MAX_ITER + 1  # load .pth
cfg.DATALOADER.NUM_WORKERS = 2


cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 2.5e-4
cfg.SOLVER.WARMUP_ITERS = 200
cfg.SOLVER.WARMUP_FACTOR = 0.4       # starts at 1e-4, ramps to 2.5e-4
cfg.SOLVER.STEPS = ()                 # no decay steps
cfg.SOLVER.GAMMA = 1.0               # no-op since no steps, but explicit is cleaner
cfg.MODEL.ROI_MASK_HEAD.RESOLUTION = 56

cfg.SOLVER.MAX_ITER = 8000
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 2

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)


trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)

ap_hook = APHook(cfg, "[your_sample_name]_valid", period=200)
early_stop = EarlyStoppingHook(ap_hook, patience=5, period=200, min_delta=0.001)

trainer.register_hooks([ap_hook, early_stop])

try:
    trainer.train()
except StopIteration:
    print("Training stopped early ✅")

